# Formula 1 Race Strategy Simulator
## Notebook 03: Sensitivity Sweeps & Decision Phase Boundaries

This notebook explores **Phase 4**:
1. **1D Sensitivity Sweeps**: Continuous sweeps of pit lane transit loss $t_{loss} \in [15.0\text{s}, 35.0\text{s}]$ and tyre degradation multiplier $\mu_{deg} \in [0.5, 2.5]$.
2. **Crossover Tipping Point Root-Finding**: Exact determination of threshold parameters via Brent's method.
3. **2D Decision Phase Boundaries**: Isoline contour mapping of $\Delta T(t_{pit}, \mu_{deg}) = 0$ separating 1-stop and 2-stop dominance regimes.
4. **Circuit Strategy Profiles**: Comparing high-degradation Bahrain with low-degradation Monza.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.model import RaceModel
from src.config import BAHRAIN_CONFIG, MONZA_CONFIG, DEFAULT_COMPOUNDS
from src.strategies import Strategy, Stint
from src.sensitivity import SensitivityAnalyzer

sns.set_theme(style="darkgrid")
print("Sensitivity analyzer ready.")

### 1. 1D Sensitivity: Pit Lane Time Loss Sweep

How sensitive is optimal strategy selection to the pit lane time loss?
We evaluate the delta between 1-Stop and 2-Stop:
$$\Delta T(t_{pit}) = T(S_{\text{1-stop}}; t_{pit}) - T(S_{\text{2-stop}}; t_{pit})$$
* If $\Delta T > 0$: 2-Stop is faster.
* If $\Delta T < 0$: 1-Stop is faster.
* Root $\Delta T(t_{pit}^*) = 0$ identifies the exact tipping point.

In [ ]:
model = RaceModel(BAHRAIN_CONFIG)
analyzer = SensitivityAnalyzer(model)

strat_1stop = Strategy([Stint(DEFAULT_COMPOUNDS["Soft"], 18), Stint(DEFAULT_COMPOUNDS["Hard"], 39)], name="1-Stop (S-H)")
strat_2stop = Strategy([Stint(DEFAULT_COMPOUNDS["Soft"], 15), Stint(DEFAULT_COMPOUNDS["Medium"], 21), Stint(DEFAULT_COMPOUNDS["Medium"], 21)], name="2-Stop (S-M-M)")

pit_losses = np.linspace(15.0, 32.0, 35)
curve = analyzer.sweep_pit_loss(strat_1stop, strat_2stop, pit_losses)

# Find exact crossover tipping point using Brent's method
crossover = analyzer.find_crossover(strat_1stop, strat_2stop, "pit_loss", bounds=(15.0, 32.0))
if crossover.crossover_value is not None:
    print(f"Crossover Tipping Point: t_pit = {crossover.crossover_value:.2f} s")
else:
    print("No crossover observed in given bounds.")

### 2. Plotting 1D Crossover Curve

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(pit_losses, curve.delta_times, color="purple", linewidth=2.5, label=r"$\Delta T = T_{1stop} - T_{2stop}$")
plt.axhline(0, color="gray", linestyle="--", alpha=0.7)

if crossover.crossover_value is not None:
    plt.axvline(crossover.crossover_value, color="red", linestyle=":", label=f"Tipping Point: {crossover.crossover_value:.2f}s")
    plt.scatter([crossover.crossover_value], [0], color="red", s=80, zorder=5)

plt.fill_between(pit_losses, curve.delta_times, 0, where=(curve.delta_times > 0), color="orange", alpha=0.2, label="2-Stop Dominates")
plt.fill_between(pit_losses, curve.delta_times, 0, where=(curve.delta_times < 0), color="blue", alpha=0.2, label="1-Stop Dominates")

plt.title("1D Pit Loss Sensitivity & Strategy Crossover: Bahrain GP", fontsize=13, fontweight="bold")
plt.xlabel("Pit Loss (seconds)", fontsize=11)
plt.ylabel("Time Delta: 1-Stop - 2-Stop (s)", fontsize=11)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 3. 2D Decision Phase Boundaries

In competitive racing, pit loss and track degradation fluctuate simultaneously (e.g. wet weather transition, green track evolution, or hot track temps).
The 2D decision boundary is defined by the level set:
$$\mathcal{B} = \big\{ (t_{pit}, \mu_{deg}) \in \mathbb{R}^2 \mid T(S_1; t_{pit}, \mu_{deg}) - T(S_2; t_{pit}, \mu_{deg}) = 0 \big\}$$

In [ ]:
phase_res = analyzer.compute_phase_diagram_2d(
    strat_1stop,
    strat_2stop,
    pit_loss_range=(16.0, 30.0),
    deg_range=(0.7, 1.8),
    grid_res=30
)

plt.figure(figsize=(10, 6))
cp = plt.contourf(phase_res.deg_grid, phase_res.pit_grid, phase_res.delta_grid, levels=25, cmap="RdYlBu_r")
cbar = plt.colorbar(cp)
cbar.set_label("Time Advantage: 2-Stop minus 1-Stop (s)")

# Decision boundary contour line
cs = plt.contour(phase_res.deg_grid, phase_res.pit_grid, phase_res.delta_grid, levels=[0.0], colors="black", linewidths=2.5, linestyles="--")
plt.clabel(cs, inline=True, fmt="Decision Boundary (0s)", fontsize=11)

nom_deg, nom_pit = phase_res.nominal_point
plt.scatter([nom_deg], [nom_pit], color="lime", edgecolor="black", s=150, zorder=10, label=f"Nominal Bahrain Point ({nom_deg}x, {nom_pit}s)")

plt.title("2D Strategic Phase Boundary: Pit Loss vs Tyre Degradation", fontsize=14, fontweight="bold")
plt.xlabel("Tyre Degradation Multiplier", fontsize=12)
plt.ylabel("Pit Loss (s)", fontsize=12)
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 4. Cross-Circuit Comparison: Monza vs Bahrain

Why does Monza favor a 1-stop strategy while Bahrain strongly favors a 2-stop?
Let us run the DP optimizer on both circuits.

In [ ]:
from src.optimization import StrategyOptimizer

monza_model = RaceModel(MONZA_CONFIG)
optimizer_bahrain = StrategyOptimizer(model)
optimizer_monza = StrategyOptimizer(monza_model)

opt_bah = optimizer_bahrain.optimize_dp(max_stops=2)
opt_mon = optimizer_monza.optimize_dp(max_stops=2)

print("=== CIRCUIT STRATEGY COMPARISON ===")
print(f"Bahrain: {model.circuit.name} ({model.circuit.total_laps} laps, pit loss: {model.pitstop_model.pit_loss}s)")
print(f"  Optimal: {opt_bah.optimal_strategy.name}")
print(f"
Monza: {monza_model.circuit.name} ({monza_model.circuit.total_laps} laps, pit loss: {monza_model.pitstop_model.pit_loss}s)")
print(f"  Optimal: {opt_mon.optimal_strategy.name}")